# 6. Pandas data work

Standalone notebook: it includes its own imports and runs top to bottom with no input prompts. Each `assert` raises an error if something is wrong, so no error means the section works.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

## Cleaning messy data

Steps: drop duplicate rows, tidy city text, convert scores to numbers (bad values become `NaN`),
parse dates, and fill missing hours with the median.

For dates, `pd.to_datetime(..., errors="coerce")` turns anything that doesn't match the first
format into `NaT`, which would lose `27-09-2026`. Two explicit formats are tried instead.

In [ ]:
raw = pd.DataFrame({
    "name": ["Ali", "Sara", "Amina", "Amina", "Usman", "Mustafa"],
    "city": [" lahore", "KARACHI", "Lahore ", "Lahore ", "Islamabad", "Sahiwal"],
    "hours": [2, np.nan, 4, 4, 3, 6],
    "score": ["55", "72", "missing", "missing", "64", "37"],
    "date": ["2026-09-01", "2026-09-02", "bad date", "bad date", "2026-09-05", "27-09-2026"],
})
print(raw.isna().sum())
print("Duplicate rows:", raw.duplicated().sum())

In [ ]:
def parse_dates(series: pd.Series) -> pd.Series:
    """Parse YYYY-MM-DD first, then DD-MM-YYYY; anything else becomes NaT."""
    iso = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
    day_first = pd.to_datetime(series, format="%d-%m-%Y", errors="coerce")
    return iso.fillna(day_first)


def clean_students(raw: pd.DataFrame) -> pd.DataFrame:
    clean = raw.drop_duplicates().copy()
    clean["city"] = clean["city"].str.strip().str.title()
    clean["score"] = pd.to_numeric(clean["score"], errors="coerce")
    clean["date"] = parse_dates(clean["date"])
    clean["hours"] = clean["hours"].fillna(clean["hours"].median())
    return clean


clean = clean_students(raw)
clean

In [ ]:
assert len(clean) == 5
assert list(clean["city"]) == ["Lahore", "Karachi", "Lahore", "Islamabad", "Sahiwal"]
assert clean["score"].isna().sum() == 1                      # "missing" -> NaN
assert clean["hours"].isna().sum() == 0
assert clean["hours"].iloc[1] == 3.5                         # median of 2, 4, 3, 6
assert clean["date"].isna().sum() == 1                       # "bad date"
assert clean["date"].iloc[-1] == pd.Timestamp("2026-09-27")  # DD-MM-YYYY recovered

## Combining tables (`concat`)

In [ ]:
term1 = pd.DataFrame({"name": ["Ali", "Sara"], "score": [55, 72]})
term2 = pd.DataFrame({"name": ["Amina", "Usman"], "score": [68, 64]})

combined = pd.concat([term1, term2], ignore_index=True)
combined

In [ ]:
assert len(combined) == 4
assert list(combined.index) == [0, 1, 2, 3]

## Long and wide formats (`pivot` and `melt`)

In [ ]:
long = pd.DataFrame({
    "name": ["Ali", "Ali", "Sara", "Sara"],
    "subject": ["Math", "AI", "Math", "AI"],
    "score": [70, 82, 88, 91],
})
wide = long.pivot(index="name", columns="subject", values="score")
back_to_long = wide.reset_index().melt(id_vars="name", value_name="score")

print(long, "\n")
print(wide, "\n")
print(back_to_long)

In [ ]:
assert wide.loc["Ali", "AI"] == 82 and wide.loc["Sara", "Math"] == 88
assert len(back_to_long) == len(long)

## Leads data

The original read `C:\Users\hp\Downloads\leads.csv`, which only exists on your machine. The code now
looks for the file at `LEADS_PATH`. If it isn't found, it uses a small built-in sample with the
columns the original code used (`Priority`, `Status`, `Date`), so the cell still runs.

Set `LEADS_PATH` to your real file to use your data.

In [ ]:
LEADS_PATH = Path(r"C:\Users\hp\Downloads\leads.csv")   # change to your file location
REQUIRED_COLUMNS = {"Priority", "Status", "Date"}


def sample_leads() -> pd.DataFrame:
    return pd.DataFrame({
        "Lead_ID": [1, 2, 3, 4, 5],
        "Priority": ["High", "Medium", "Low", "High", "Medium"],
        "Status": ["New", "Contacted", "New", "Qualified", "Lost"],
        "Date": ["2026-09-01", "2026-09-03", "2026-09-04", "2026-09-07", "2026-09-10"],
    })


def load_leads(path: Path = LEADS_PATH):
    """Return (dataframe, using_real_file)."""
    if path.exists():
        return pd.read_csv(path), True
    return sample_leads(), False


leads, from_file = load_leads()
print("Loaded from file." if from_file else "leads.csv not found: using the built-in sample.")
print(leads)

In [ ]:
if REQUIRED_COLUMNS <= set(leads.columns):
    # Select rows by priority and only the Status and Date columns
    by_priority = leads.set_index("Priority").loc[["Medium", "High"], ["Status", "Date"]]
    print(by_priority)

    # New column combining status and date
    leads = leads.copy()
    leads["New_Column"] = leads["Status"] + " - " + leads["Date"].astype(str)
    print()
    print(leads["New_Column"])
else:
    print("Skipped: the file needs these columns:", sorted(REQUIRED_COLUMNS))

## Superstore sales cleaning and analysis pipeline

The pipeline is split into functions: `clean_superstore`, `summarize`, and `build_analysis`.

The original read `sampledata1.csv`. If that file is next to the notebook it is used; otherwise a
small **synthetic** dataset with the same columns (including a few deliberately bad rows) is generated,
so the pipeline can be shown end to end. Results from the synthetic data are not meaningful.

In [ ]:
SUPERSTORE_PATH = Path("sampledata1.csv")


def make_sample_superstore(n: int = 150, seed: int = 42) -> pd.DataFrame:
    """Synthetic data with the Superstore columns and a few intentionally bad rows."""
    rng = np.random.default_rng(seed)
    order = pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 700, n), unit="D")
    ship = order + pd.to_timedelta(rng.integers(1, 8, n), unit="D")
    sales = rng.uniform(10, 2000, n).round(2)

    df = pd.DataFrame({
        "Row ID": np.arange(1, n + 1),
        "Order ID": [f"CA-{i:05d}" for i in range(n)],
        "Order Date": order.strftime("%Y-%m-%d"),
        "Ship Date": ship.strftime("%Y-%m-%d"),
        "Ship Mode": rng.choice(["Standard Class", "Second Class", "First Class"], n),
        "Customer Name": rng.choice(["Ali Khan", "Sara Ahmed", "Hamza Ali", "Noor Fatima", "Bilal Raza"], n),
        "Segment": rng.choice(["Consumer", "Corporate", "Home Office"], n),
        "State": rng.choice(["Punjab", "Sindh", "Balochistan"], n),
        "Postal Code": rng.integers(10000, 99999, n),
        "Region": rng.choice(["East", "West", "Central"], n),
        "Product ID": [f"P-{i}" for i in rng.integers(1, 30, n)],
        "Category": rng.choice(["Furniture", "Technology", "Office Supplies"], n),
        "Sub-Category": rng.choice(["Chairs", "Phones", "Paper", "Binders"], n),
        "Product Name": rng.choice(["Chair A", "Phone B", "Paper C", "Binder D"], n),
        "Sales": sales,
        "Quantity": rng.integers(1, 10, n),
        "Discount": rng.choice([0, 0.1, 0.2, 0.4, 0.5], n),
        "Profit": (sales * rng.uniform(-0.3, 0.4, n)).round(2),
    })
    df.loc[0, "Customer Name"] = "  Ali Khan "                      # stray spaces
    df.loc[3, "Order Date"] = None                                  # missing date
    df.loc[5, "Discount"] = 1.5                                     # impossible discount
    df.loc[7, "Ship Date"] = (order[7] - pd.Timedelta(days=2)).strftime("%Y-%m-%d")  # ships before ordered
    duplicates = df.iloc[[10, 11]].copy()
    duplicates["Row ID"] = [n + 1, n + 2]                           # exact duplicates apart from Row ID
    return pd.concat([df, duplicates], ignore_index=True)


def load_superstore(path: Path = SUPERSTORE_PATH):
    """Return (dataframe, using_real_file)."""
    if path.exists():
        return pd.read_csv(path, encoding="latin-1"), True
    return make_sample_superstore(), False


raw_sales, from_file = load_superstore()
print("Loaded real file." if from_file else "sampledata1.csv not found: using synthetic sample data.")
print(raw_sales.shape)

In [ ]:
def iqr_outliers(series: pd.Series) -> pd.Series:
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)


def clean_superstore(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    df = df.copy()

    # Standardise column names: "Sub-Category" -> "sub_category"
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(" ", "_").str.replace("-", "_"))

    # Types
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")
    df["postal_code"] = df["postal_code"].astype("string")      # an ID, not a number

    # Trim whitespace in every text column
    text_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
    df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())

    for col in ["ship_mode", "segment", "region", "category", "sub_category"]:
        df[col] = df[col].astype("category")

    # Duplicates (ignoring the row id) and missing key fields
    subset = [c for c in df.columns if c != "row_id"]
    n_dupes = df.duplicated(subset=subset).sum()
    df = df.drop_duplicates(subset=subset, keep="first")
    df = df.dropna(subset=["order_id", "order_date", "sales"])

    # Logical checks
    bad_dates = df["ship_date"] < df["order_date"]
    bad_discount = ~df["discount"].between(0, 1)
    bad_qty = df["quantity"] <= 0
    if verbose:
        print(f"Exact duplicates removed: {n_dupes}")
        print(f"Ship before order: {bad_dates.sum()} | bad discount: {bad_discount.sum()} | bad quantity: {bad_qty.sum()}")
    df = df[~(bad_dates | bad_discount | bad_qty)].copy()

    # Derived columns
    df["is_profit_outlier"] = iqr_outliers(df["profit"])
    df["year"] = df["order_date"].dt.year
    df["month"] = df["order_date"].dt.to_period("M").astype(str)
    df["ship_days"] = (df["ship_date"] - df["order_date"]).dt.days
    df["margin_pct"] = np.where(df["sales"] > 0, df["profit"] / df["sales"] * 100, 0)
    df["is_loss"] = df["profit"] < 0
    df["discount_band"] = pd.cut(
        df["discount"], bins=[-0.01, 0, 0.2, 0.4, 1],
        labels=["None", "Low (≤20%)", "Medium (≤40%)", "High (>40%)"],
    )
    return df


sales_clean = clean_superstore(raw_sales)
print(sales_clean.shape)
sales_clean.head()

In [ ]:
def summarize(df: pd.DataFrame, by) -> pd.DataFrame:
    return (df.groupby(by, observed=True)
              .agg(sales=("sales", "sum"),
                   profit=("profit", "sum"),
                   orders=("order_id", "nunique"),
                   qty=("quantity", "sum"),
                   avg_discount=("discount", "mean"))
              .assign(margin_pct=lambda x: x["profit"] / x["sales"] * 100)
              .sort_values("sales", ascending=False)
              .round(2))


def build_analysis(df: pd.DataFrame) -> dict:
    yearly = df.groupby("year")[["sales", "profit"]].sum()
    yearly["sales_growth_%"] = yearly["sales"].pct_change() * 100
    return {
        "by_category": summarize(df, "category"),
        "by_subcategory": summarize(df, ["category", "sub_category"]),
        "by_region": summarize(df, "region"),
        "by_segment": summarize(df, "segment"),
        "by_state": summarize(df, "state"),
        "by_ship_mode": summarize(df, "ship_mode"),
        "by_month": summarize(df, "month").sort_index(),
        "by_discount": summarize(df, "discount_band"),   # shows how discounts affect profit
        "profit_heatmap": df.pivot_table(index="region", columns="category",
                                         values="profit", aggfunc="sum"),
        "top_customers": summarize(df, "customer_name").head(10),
        "loss_products": summarize(df, "product_name").sort_values("profit").head(10),
        "correlation": df[["sales", "quantity", "discount", "profit"]].corr(),
        "yearly": yearly,
    }


def save_clean(df: pd.DataFrame, stem: str = "clean_superstore") -> None:
    """Write the cleaned data to CSV, and to Parquet if pyarrow is installed."""
    df.to_csv(f"{stem}.csv", index=False)
    try:
        df.to_parquet(f"{stem}.parquet", index=False)
    except ImportError:
        print("Parquet skipped: install pyarrow to enable it.")


analysis = build_analysis(sales_clean)
print(analysis["by_category"], "\n")
print(analysis["by_discount"], "\n")
print(analysis["correlation"], "\n")
print(analysis["yearly"])

# save_clean(sales_clean)   # uncomment to write clean_superstore.csv / .parquet

In [ ]:
assert len(sales_clean) < len(raw_sales)
assert (sales_clean["ship_date"] >= sales_clean["order_date"]).all()
assert sales_clean["discount"].between(0, 1).all()
assert sales_clean["order_date"].notna().all()
assert (sales_clean["ship_days"] >= 0).all()
assert abs(analysis["by_category"]["sales"].sum() - sales_clean["sales"].sum()) < 1

In [ ]:
print("Pandas data work: all tests passed.")